In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun, ShellTool 
from langchain_core.tools import StructuredTool, BaseTool,tool
from pydantic import BaseModel, Field
from langchain_core.messages import ToolMessage
from typing import Type
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

C:\Users\kmpra\AppData\Local\Temp\ipykernel_21352\1959961953.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun, ShellTool


# Built In Tools 

In [2]:

# Searh Tool
search_tool = DuckDuckGoSearchRun()
result = search_tool.invoke("Today's news")
print(result)

print('*'*100)

# Command Tool
command_tool = ShellTool()
result = command_tool.invoke("whoami")
print(result)

print('*'*100)

print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

Rating 4.2 (6,21,236) · Free · Android 5 days ago · India's most trusted English news app. Get breaking news alerts, current events,today news, Read today's English news across 71 cities in 14 languages. 3 hours ago · Union Home Minister Amit Shah today chaired the 28th meeting of the Western Zonal Council at Panaji in Goa. Addressing the gathering, Mr Shah said, there ... 4 hours ago · Today Breaking News ! आज 08 सितंबर 2026 के मुख्य समाचार बड़ी खबरें, PM Modi, UP, Bihar, Delhi, SBI. 2 hours ago · News · At least five killed after Amazon cargo plane crash in Miami airport · Egyptian TV presenter sentenced to death in drugs case · Panama Canal may cut ship ... 1 hour ago · Delhi building collapse death toll hits 7; HC orders citywide PG inspection, flags student housing woes · Natural partners: On India-Belgium ties · Banned ...
****************************************************************************************************
Executing command:
 whoami
zero_book_13\kmpra

***********

E:\AI-Engineering\Practicals\.venv\lib\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# Custom Tools

## Tool decorator

In [3]:
# Addition tools
# tool decorator + defination + doc comment for better understandings
@tool
def add_num(a:int, b:int) -> int:
    '''Addition of Two number'''
    return a+b

# Tool calling
print(add_num.invoke({"a":2, "b":3}))

print(add_num.name)
print(add_num.description)
print(add_num.args)

5
add_num
Addition of Two number
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Structured Tool

In [4]:
# Give structure with pydantic schemas

class MultiplyInput(BaseModel):
    a:int = Field(description="first number to multiply")
    b:int = Field(description="second number to multiply")

def multiply_fun(a:int, b:int) -> int:
    return a*b

multiply_tool = StructuredTool.from_function(
    func=multiply_fun,
    name="multipy",
    description="multiplication of Two number",
    args_schema=MultiplyInput
)

print(multiply_tool.invoke({"a":2, "b":3}))
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

6
multipy
multiplication of Two number
{'a': {'description': 'first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'second number to multiply', 'title': 'B', 'type': 'integer'}}


## BaseTool

In [5]:
# pydantic + BaseTool
# BaseTool is parent of other two(decorator, SStructuredTool)

class MultiplyInput(BaseModel):
    a:int = Field(description="first number to multiply")
    b:int = Field(description="second number to multiply")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiplication of two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a:int, b:int) -> int:
        return a*b

multiply_tool = MultiplyTool()

print(multiply_tool.invoke({"a":2, "b":3}))
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

6
multiply
Multiplication of two numbers
{'a': {'description': 'first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'second number to multiply', 'title': 'B', 'type': 'integer'}}


# Tool binding and calling

In [22]:
# tool binding
llm_with_tool = llm.bind_tools([add_num, multiply_tool])

# tool calling
response = llm_with_tool.invoke("What is addition of  10 with 20?")

tool = response.tool_calls

response.tool_calls

[{'name': 'add_num',
  'args': {'a': 10, 'b': 20},
  'id': 'fc_32b2c330-5b76-4a98-afad-36512c9b3643',
  'type': 'tool_call'}]

# Tool Execution

In [23]:
tools = [add_num, multiply_tool]

# 3. Map tools
tools_by_name = {
    tool.name: tool
    for tool in tools
}


# 4. Execute each tool call
tool_messages = []

for tool_call in response.tool_calls:

    tool = tools_by_name[tool_call["name"]]

    result = tool.invoke(tool_call)
 
    tool_messages.append(result)


# 5. Send tool result back to LLM
final_response = llm_with_tool.invoke(
    [
        response,
        *tool_messages
    ]
)

print(final_response.content)

The sum of 10 and 20 is **30**.
